# ساخت نگاشت موجودیت‌ها (Entities) و روابط (Relations) به فرمت JSON

این نوت‌بوک دو کار انجام می‌دهد:

1. از روی فایل `entity.csv` یک فایل JSON با فرمت `{"m.<freebase_id>": "label, description"}` می‌سازد.
2. از روی فایل `relation.txt` یک فایل JSON با فرمت `{"relation_name": "relation_name"}` می‌سازد (هر رابطه به مقدار خودش نگاشت می‌شود).

**نکته:** مسیر فایل‌های ورودی و خروجی را در سلول تنظیمات زیر مطابق نیاز خودتان تغییر دهید.

In [17]:
import pandas as pd
import json
import os

# ---------------------- تنظیمات مسیرها ----------------------
ENTITY_CSV_PATH   = "entities_data.csv"              # مسیر فایل ورودی موجودیت‌ها
RELATION_CSV_PATH = "relations_data.csv"            # مسیر فایل ورودی روابط

ENTITY_JSON_OUT   = "entity_text.json"     # خروجی نگاشت موجودیت‌ها
RELATION_JSON_OUT = "relation_text.json"   # خروجی نگاشت روابط


## ۱. ساخت نگاشت موجودیت‌ها از `entity.csv`

In [18]:
# فایل با تب (\t) از هم جدا شده است
df = pd.read_csv(ENTITY_CSV_PATH, sep=",", dtype=str, keep_default_na=False)

print(df.shape)
df.head()


(48551, 8)


,id,label,description,has_page,has_data,images,image_count,url
0,Q1,universe,"totality consisting of space, time, matter and...",TRUE,TRUE,caption=NASA-HS201427a-HubbleUltraDeepField201...,3,https://www.wikidata.org/wiki/Q1
1,Q100,Boston,capital city of the U.S. state of Massachusett...,TRUE,TRUE,"caption=Boston from the Harbour, Massachusetts...",1,https://www.wikidata.org/wiki/Q100
2,Q1000,Gabon,country on the Atlantic coast of Central Africa,TRUE,TRUE,caption=Topographic map of Gabon-fr.svg ; comm...,1,https://www.wikidata.org/wiki/Q1000
3,Q1000118,Peter Pettigrew,fictional character of the Harry Potter series,TRUE,TRUE,,0,https://www.wikidata.org/wiki/Q1000118
4,Q1000136,Speedway,"town in Wayne Township, Marion County, Indiana",TRUE,TRUE,caption=Indianapolis-motor-speedway.jpg ; comm...,1,https://www.wikidata.org/wiki/Q1000136


In [19]:
# def make_key(freebase_id: str) -> str:
#     """
#     freebase_id در فایل csv به شکل '0100gqqp' است.
#     فرمت استاندارد Freebase mid به صورت 'm.0100gqqp' است،
#     پس در صورتی که پیشوند 'm.' وجود نداشته باشد، اضافه می‌شود.
#     """
#     freebase_id = str(freebase_id).strip()
#     if not freebase_id:
#         return freebase_id
#     if freebase_id.startswith("m.") or freebase_id.startswith("/m/"):
#         return freebase_id.replace("/m/", "m.")
#     return f"{freebase_id}"


def clean_text(text: str) -> str:
    """
    بعضی از متن‌ها در فایل csv به‌خاطر escape شدن کوتیشن‌های داخلی،
    به‌صورت "" (دو کوتیشن پشت‌سرهم) نوشته شده‌اند که باید به یک
    کوتیشن معمولی (") تبدیل شوند. مثال:
        'title character of ""Phineas and Ferb""'
        -> 'title character of "Phineas and Ferb"'
    """
    text = (text or "").strip()
    text = text.replace('""', '"')
    return text


def make_value(label: str, description: str) -> str:
    """
    مقدار value از اتصال label و description ساخته می‌شود.
    اگر هرکدام خالی بودند، آن قسمت حذف می‌شود.
    اگر هر دو خالی بودند، رشته خالی برگردانده می‌شود.
    """
    label = clean_text(label)
    description = clean_text(description)
    
    parts = [p for p in (label, description) if p]
    if not parts:
        return ""
    return ", ".join(parts)


In [20]:
entity_map = {}
for _, row in df.iterrows():
    key = row["id"]
    if not key:
        continue
    value = make_value(row.get("label", ""), row.get("description", ""))
    entity_map[key] = value

print(f"تعداد موجودیت‌های ساخته‌شده: {len(entity_map)}")
# نمایش چند نمونه
list(entity_map.items())[:5]

تعداد موجودیت‌های ساخته‌شده: 48551


[('Q1', 'universe, totality consisting of space, time, matter and energy'),
 ('Q100',
  'Boston, capital city of the U.S. state of Massachusetts and seat of Suffolk County'),
 ('Q1000', 'Gabon, country on the Atlantic coast of Central Africa'),
 ('Q1000118',
  'Peter Pettigrew, fictional character of the Harry Potter series'),
 ('Q1000136', 'Speedway, town in Wayne Township, Marion County, Indiana')]

In [21]:
with open(ENTITY_JSON_OUT, "w", encoding="utf-8") as f:
    json.dump(entity_map, f, ensure_ascii=False, indent=2)

print(f"فایل ذخیره شد: {ENTITY_JSON_OUT}")


فایل ذخیره شد: entity_text.json


## ۲. ساخت نگاشت روابط از `relation_data.csv`

In [22]:
# فایل با تب (\t) از هم جدا شده است
df = pd.read_csv(RELATION_CSV_PATH, sep=",", dtype=str, keep_default_na=False)

print(df.shape)
df.head()


(535, 8)


,id,label,description,has_page,has_data,images,image_count,url
0,P1001,applies to jurisdiction,"the item (institution, law, public office, pub...",True,True,,0,https://www.wikidata.org/wiki/Property:P1001
1,P101,field of work,specialization of a person or organization; se...,True,True,,0,https://www.wikidata.org/wiki/Property:P101
2,P1011,excluding,usually used as a qualifier,True,True,,0,https://www.wikidata.org/wiki/Property:P1011
3,P1012,including,usually used as a qualifier,True,True,,0,https://www.wikidata.org/wiki/Property:P1012
4,P1013,criterion used,property by which a distinction or classificat...,True,True,,0,https://www.wikidata.org/wiki/Property:P1013


In [23]:
relation_map = {}
for _, row in df.iterrows():
    key = row["id"]
    if not key:
        continue
    value = make_value(row.get("label", ""), row.get("description", ""))
    relation_map[key] = value

print(f"تعداد روابط ساخته‌شده: {len(relation_map)}")
# نمایش چند نمونه
list(relation_map.items())[:5]

تعداد روابط ساخته‌شده: 535


[('P1001',
  'applies to jurisdiction, the item (institution, law, public office, public register, etc) or statement belongs to or has power over or applies to the value (a territorial jurisdiction: a country, state, municipality, etc)'),
 ('P101',
  'field of work, specialization of a person or organization; see P106 for the occupation/ P452 for industry'),
 ('P1011', 'excluding, usually used as a qualifier'),
 ('P1012', 'including, usually used as a qualifier'),
 ('P1013',
  'criterion used, property by which a distinction or classification is made')]

In [24]:
with open(RELATION_JSON_OUT, "w", encoding="utf-8") as f:
    json.dump(relation_map, f, ensure_ascii=False, indent=2)

print(f"فایل ذخیره شد: {RELATION_JSON_OUT}")

فایل ذخیره شد: relation_text.json


### توضیح ساختار خروجی‌ها

**entity_id2text.json**
```json
{
  "m.0100gqqp": "Grimm, season 4, season of television series",
  "m.01010z9z": "Bellator Fighting Championships, Bella Fighting Championships is a TV program."
}
```

**relation_id2text.json**
```json
{
  "american_football.player_receiving_statistics": "american_football.player_receiving_statistics",
  "architecture.occupancy": "architecture.occupancy"
}
```
